In [8]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
conn = sqlite3.connect("base_tratada.db")

query = """
SELECT *
FROM base_tratada;
"""

df = pd.read_sql(query, conn)

# Visualizar os dados
df.head()

#testando se o banco conectou

,preco,hospedes,quartos,camas,tipo_de_quarto,nota_avaliacao,noites_minimas,quantidade_avaliacoes,tipo_de_propriedade,latitude,...,banheiros,tipo_quarto_casa_apto_inteiro,tipo_quarto_quarto_compartilhado,tipo_quarto_quarto_hotel,tipo_quarto_quarto_privativo,tipo_de_propriedade_grupo,bairro_encode,ano_mes,ipca_mensal,ipca_acumulado_ano
0,580.0,4,2.0,3.0,casa_apto_inteiro,4.93,2,86,entire condo,-22.982818,...,1.5,1,0,0,0,1,76,2025-09-01 00:00:00,0.0048,0.0426
1,1900.0,2,1.0,1.0,casa_apto_inteiro,NaN,5,0,entire rental unit,-22.984090,...,2.0,1,0,0,0,1,34,2025-09-01 00:00:00,0.0048,0.0426
2,700.0,4,1.0,1.0,casa_apto_inteiro,NaN,1,0,entire rental unit,-22.814911,...,1.0,1,0,0,0,1,96,2025-09-01 00:00:00,0.0048,0.0426
3,NaN,2,NaN,NaN,quarto_privativo,5.00,2,3,private room in rental unit,-22.981910,...,1.0,0,0,0,1,4,76,2025-09-01 00:00:00,0.0048,0.0426
4,500.0,4,1.0,1.0,casa_apto_inteiro,4.91,2,11,entire rental unit,-23.010000,...,1.0,1,0,0,0,1,8,2025-09-01 00:00:00,0.0048,0.0426


In [10]:
df_tratado = df[df["preco"] <= 7000].copy()

#Limita o preco das diaria ate 7k

In [11]:
df_tratado_price_min = df_tratado[
    (df_tratado["preco"] >= 65) &
    (df_tratado["quartos"] <= 15) &
    (df_tratado["banheiros"] <= 12)
].copy()

#atribui ao novo dataframe valores de limete pós remocao de outliers

In [12]:
df_tratado_price_min[
    [
        "preco",
        "hospedes",
        "quartos",
        "camas",
        "nota_avaliacao",
        "noites_minimas",
        "quantidade_avaliacoes",
        "latitude",
        "longitude",
        "banheiros",
        "tipo_de_quarto"
    ]
].describe()
df_tratado_price_min.info()

<class 'pandas.DataFrame'>
Index: 37711 entries, 0 to 43067
Data columns (total 24 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   preco                             37711 non-null  float64
 1   hospedes                          37711 non-null  int64  
 2   quartos                           37711 non-null  float64
 3   camas                             37700 non-null  float64
 4   tipo_de_quarto                    37711 non-null  str    
 5   nota_avaliacao                    30097 non-null  float64
 6   noites_minimas                    37711 non-null  int64  
 7   quantidade_avaliacoes             37711 non-null  int64  
 8   tipo_de_propriedade               37711 non-null  str    
 9   latitude                          37711 non-null  float64
 10  longitude                         37711 non-null  float64
 11  bairro                            37711 non-null  str    
 12  ano                 

In [ ]:
variaveis = [
  "preco",
    "quartos",
    "banheiros",
    "latitude",
    "longitude",
    "camas",
    "nota_avaliacao",
    "quantidade_avaliacoes",
    "hospedes",
    "noites_minimas",
    "tipo_quarto_casa_apto_inteiro",
    "tipo_quarto_quarto_compartilhado",
    "tipo_quarto_quarto_hotel",
    "tipo_quarto_quarto_privativo",
    "tipo_de_propriedade_grupo",
    "ipca_mensal",
    "ipca_acumulado_ano"
]

df_corr = df_tratado_price_min[variaveis]

In [14]:
corr_matrix = df_corr.corr()

ValueError: could not convert string to float: 'casa_apto_inteiro'

In [ ]:
plt.figure(figsize=(12, 9))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5
)

plt.title("Matriz de Correlação das Variáveis com o Preço da Diária")
plt.show()

Treinando algoritmo


### Modelo sem transformação log (baseline)

In [ ]:
# Base final definida a partir da análise exploratória

df_modelo = df_tratado_price_min.copy()


In [ ]:
# Ajuste do preço pela inflação (IPCA)

df_modelo["preco_ajustado"] = (
    df_modelo["preco"] * (1 + df_modelo["ipca_acumulado_ano"])
)

In [ ]:
# Definição do target

y = df_modelo["preco_ajustado"]

In [ ]:
# Definição das variáveis explicativas conforme decisão do grupo

X = df_modelo.drop(
    columns=[
        "preco",
        "preco_ajustado",
        "ano_mes",
        "bairro_encode",
        "bairro",
        "tipo_de_propriedade",
        "tipo_de_quarto",
        "ano",
        "mes",
        "ipca_acumulado_ano",
        "ipca_mensal"
    ]
)

In [ ]:
# Quantidade de valores nulos por coluna

df_modelo.isna().sum()

In [ ]:
# Remoção de valores ausentes

df_modelo = df_modelo.dropna()

In [ ]:
# Target

y = df_modelo["preco_ajustado"]

In [ ]:
# Features numéricas finais (decisão do grupo)

X = df_modelo.drop(
    columns=[
        "preco",
        "preco_ajustado",
        "ano_mes",
        "bairro_encode",
        "bairro",
        "tipo_de_propriedade",
        "tipo_de_quarto",
        "ano",
        "mes",
        "ipca_acumulado_ano",
        "ipca_mensal"
    ]
)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

modelo_baseline = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

modelo_baseline.fit(X_train, y_train)

In [ ]:
y_pred = modelo_baseline.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Baseline sem log")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

### Modelo com transformação log

In [ ]:
# Transformação logarítmica do preço ajustado (target)

df_modelo["preco_log"] = np.log1p(df_modelo["preco_ajustado"])

In [ ]:
# Target log

y_log = df_modelo["preco_log"]

In [ ]:
# MESMO X do baseline (somente variáveis numéricas)

X_log = X.copy()

In [ ]:
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X_log,
    y_log,
    test_size=0.3,
    random_state=42
)

In [ ]:
modelo_log = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

modelo_log.fit(X_train_log, y_train_log)

In [ ]:
y_pred_log = modelo_log.predict(X_test_log)

In [ ]:
# Converter valores de volta para reais

y_pred_log_reais = np.expm1(y_pred_log)
y_test_reais = np.expm1(y_test_log)

In [ ]:
mae_log = mean_absolute_error(y_test_reais, y_pred_log_reais)
rmse_log = np.sqrt(mean_squared_error(y_test_reais, y_pred_log_reais))
r2_log = r2_score(y_test_reais, y_pred_log_reais)

print("Modelo com transformação log")
print("MAE:", mae_log)
print("RMSE:", rmse_log)
print("R²:", r2_log)

### Comparação entre os modelos

A tabela abaixo apresenta a comparação entre o modelo baseline (sem transformação log)
e o modelo com transformação logarítmica do preço, utilizando as métricas MAE, RMSE e R².

In [ ]:
# # Cria a tabela de comparação

# from IPython.display import display
# import pandas as pd

# comparacao = pd.DataFrame({
#     "Modelo": ["Sem log", "Com log"],
#     "MAE": [mae, mae_log],
#     "RMSE": [rmse, rmse_log],
#     "R²": [r2, r2_log]
# })

In [ ]:
# # Mostrar a tabela
# display(comparacao)

# # Comparação automática baseada no MAE
# melhor_com_log = mae_log < mae

# if melhor_com_log:
#     print("✅ O modelo com transformação log apresentou melhor desempenho em MAE.")
# else:
#     print("ℹ️ O modelo sem log apresentou desempenho semelhante ou melhor em MAE.")